In [0]:
print('helpers')

In [0]:
# Import requests for making HTTP API calls to Databricks REST endpoints
import requests
# Import json for serializing and deserializing JSON payloads
import json

In [0]:
# Catalog and schema names for IoT sensor data
CATALOG_NAME = 'iot_sensor_catalog'
print(f"""CATALOG_NAME = {CATALOG_NAME}""")
Default_schema = 'default'
print(f"""Default_schema = {Default_schema}""")
Volume_name = 'iot_sensor_volume'
print(f"""Volume_name = {Volume_name}""")

# Full path for the catalog volume
CATALOG_VOLUME = f'{CATALOG_NAME}.{Default_schema}.{Volume_name}'
print(f"""CATALOG_VOLUME = {CATALOG_VOLUME}""")

# Bronze schema and database for raw IoT data
Bronze_schema = 'Bronze_IOT'
BRONZE_DATABASE = f'{CATALOG_NAME}.{Bronze_schema}'
print(f'BRONZE_DATABASE = {BRONZE_DATABASE}')

# Silver schema and database for cleaned IoT data
Silver_schema = 'Silver_IOT'
SILVER_DATABASE = f'{CATALOG_NAME}.{Silver_schema}'

# Create the database for silver layer if it does not exist
query = f"""
CREATE DATABASE IF NOT EXISTS {SILVER_DATABASE}
COMMENT 'Database for IOT silver layer'
"""

print(query)
# Execute the SQL statement to create the database
spark.sql(query)

In [0]:
class DatabricksJobManager:
    def __init__(self, job_name, job_settings):
        # Initialize job manager with job name and settings
        self.job_name = job_name
        self.job_settings = job_settings
        # Get Databricks REST API URL and token from notebook context
        self.api_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
        self.api_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        # Set headers for REST API requests
        self.headers = {
            "Authorization": f"Bearer {self.api_token}",
            "Content-Type": "application/json"
        }

    def create_job_if_not_exists(self):
        # List all jobs to check if the job already exists
        list_jobs_response = requests.get(
            f"{self.api_url}/api/2.1/jobs/list",
            headers=self.headers
        )
        jobs = list_jobs_response.json().get("jobs", [])
        job_exists = [i['settings']['name'] for i in jobs]

        # Check if the job already exists if not then it will create in else part 
        if self.job_name in job_exists:
            # Job already exists, print message
            print(f"Job {self.job_name} already exists.")
        else:
            # Create the job using Databricks REST API
            response = requests.post(
                f"{self.api_url}/api/2.1/jobs/create",
                headers=self.headers,
                data=json.dumps(self.job_settings)
            )
            job_response = response.json()
            print('job created successfully')

In [0]:
class DatabricksPipelineManager:
    def __init__(self, pipeline_name, pipeline_payload):
        # Initialize pipeline manager with pipeline name and payload
        self.pipeline_name = pipeline_name
        self.pipeline_settings = pipeline_payload
        # Get Databricks REST API URL and token from notebook context
        self.api_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
        self.TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        
    def create_pipeline_if_not_exists(self):
        # List all pipelines to check if the pipeline already exists
        list_response = requests.get(
            f"{self.api_url}/api/2.0/pipelines",
            headers={
                "Authorization": f"Bearer {self.TOKEN}"
            }
        )
        pipelines = list_response.json().get("statuses", [])
        pipeline_exists = [i['name'] for i in pipelines]
        # Check if the pipeline already exists
        if pipeline_name in pipeline_exists:
            print(f"Pipeline {pipeline_name} already exists")
        else:
            # Create the pipeline using Databricks REST API if not then will create in else part
            response = requests.post(
                f"{self.api_url}/api/2.0/pipelines",
                headers={
                    "Authorization": f"Bearer {self.TOKEN}",
                    "Content-Type": "application/json"
                },
                json=pipeline_payload
            )
            print('pipeline created successfully')

In [0]:
# STREAM PROCESSOR CLASS
class StreamProcessor:
    def __init__(
        self,
        source_table_name,
        checkpoint_location,
        target_table,
        query
    ):
        # Initialize StreamProcessor with source table, checkpoint location, target table, and transformation query
        self.source_table_name = source_table_name
        self.checkpoint_location = checkpoint_location
        self.target_table = target_table
        self.query = query
        
    # READ STREAM
    def read_stream(self):
        # Read streaming data from the source table
        self.df = (
            spark.readStream
            .option("skipChangeCommits", "true")
            .table(self.source_table_name)
        )
        return self.df

    def process_records(self, batch_df, batch_id):
        # Placeholder for custom batch processing logic
        pass

    def process_batch(self,batch_df, batch_id):
        # NOTE: On Serverless, prints in foreachBatch don't appear in notebook output
        # They run in remote workers. Track progress via the target table instead.
        # Create temp view for batch processing
        view_name = self.source_table_name.split('.')[-1]
        temp_view = f'{view_name}_view'
        batch_df.createOrReplaceTempView(temp_view)
     
        # Transformation with MERGE using provided query
        query = f"""{self.query}
                    """.replace('temp_view',temp_view)

        result_df = spark.sql(query)

        # If target table exists, delete matching sensor_event_id records before appending new batch
        if spark.catalog.tableExists(self.target_table):
            event_ids = result_df.select('sensor_event_id').distinct().collect()
            event_ids = ','.join([f""" "{row['sensor_event_id']}" """ for row in event_ids])

            delete_query = f"""
                DELETE FROM {self.target_table}
                WHERE sensor_event_id IN ({event_ids})"""
            print(delete_query)
            spark.sql(delete_query)
       
        # Write transformed batch to target table with schema merge and change data feed enabled
        result_df.write.mode("append").option('mergeSchema', True).option("enableChangeDataFeed", "true").saveAsTable(
            self.target_table
        )

    def write_stream(
        self,
        df,
        output_mode="append",
        trigger_type='availableNow',
        processing_time=None
    ):
        # Start streaming write using foreachBatch for custom batch processing
        print("Starting foreachBatch stream write...")
        print("Note: Batch processing happens in remote workers - check target table for progress")

        writer = (
            df.writeStream
            .queryName('xyz')
            .foreachBatch(self.process_batch)
            .outputMode(output_mode)
            .option(
                "checkpointLocation",
                self.checkpoint_location
            )
        )

        # Trigger handling for stream execution mode
        if trigger_type=='availableNow':
            writer = writer.trigger(availableNow=True)
        elif trigger_type == 'once':
            writer = writer.trigger(
                once=True
            )
        else:
            writer = writer.trigger(
                processingTime=processing_time
            )

        # Start streaming query
        query = writer.start()
        
        return query